In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import zipfile
import tarfile
import time


# =============================================================================
# CONFIGURATION
# =============================================================================

ZIP_PATH = Path(
    r"C:\Users\rwen13\Desktop\consumer_panel\2004-2020_Panel_Data.zip"
)

# Print every final-layer path stored inside each TGZ.
PRINT_COMPLETE_PATHS = True

# Print a tree-like view of every final-layer path.
PRINT_COMPLETE_TREE = True

# Number of example paths shown in the summary section.
N_SAMPLE_PATHS = 30


# =============================================================================
# HARD READ-ONLY RULE
# =============================================================================
#
# THIS CELL DOES NOT:
#
#   - extract files to disk
#   - delete files
#   - rename files
#   - create folders
#   - modify the ZIP
#   - modify any TGZ
#
# It only performs:
#
#   outer ZIP
#       -> open TGZ member as a stream
#       -> decompress TGZ in memory/streaming mode
#       -> read TAR member metadata
#       -> print the final internal structure
#
# No permanent files are created.
# =============================================================================


# =============================================================================
# 1. Validate source ZIP
# =============================================================================

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP does not exist:\n{ZIP_PATH}"
    )

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        f"Not a valid ZIP archive:\n{ZIP_PATH}"
    )


# =============================================================================
# 2. Helper: identify archive-like files
# =============================================================================

ARCHIVE_SUFFIXES = (
    ".zip",
    ".tgz",
    ".tar",
    ".tar.gz",
    ".gz",
    ".bz2",
    ".xz",
    ".7z",
    ".rar",
)


def is_archive_like(name):
    """
    Identify whether a final-layer member appears to be another archive.
    """
    lower = name.lower()

    return any(
        lower.endswith(suffix)
        for suffix in ARCHIVE_SUFFIXES
    )


# =============================================================================
# 3. Helper: display a path as a tree
# =============================================================================

def print_tree_path(member_name, is_dir=False):
    """
    Print a TAR member using indentation based on path depth.
    """

    clean = member_name.strip("/")

    if not clean:
        return

    parts = clean.split("/")

    depth = len(parts) - 1
    indent = "    " * depth

    kind = "[DIR ]" if is_dir else "[FILE]"

    print(
        f"{indent}{kind} {parts[-1]}"
    )


# =============================================================================
# 4. Inspect each TGZ INSIDE the outer ZIP
# =============================================================================

overall_start = time.time()

all_archive_summaries = []

print("=" * 110)
print("DEEP READ-ONLY ARCHIVE INSPECTION")
print("OUTER ZIP -> TGZ -> FINAL TAR CONTENTS")
print("=" * 110)

print(f"\nSource ZIP:")
print(ZIP_PATH)


with zipfile.ZipFile(ZIP_PATH, "r") as outer_zip:

    # -------------------------------------------------------------------------
    # Find the TGZ files stored inside the outer ZIP.
    # -------------------------------------------------------------------------

    tgz_infos = [
        info
        for info in outer_zip.infolist()
        if (
            not info.is_dir()
            and (
                info.filename.lower().endswith(".tgz")
                or info.filename.lower().endswith(".tar.gz")
            )
        )
    ]

    tgz_infos = sorted(
        tgz_infos,
        key=lambda x: x.filename.lower()
    )

    print(
        f"\nNested TGZ archives found: "
        f"{len(tgz_infos):,}"
    )


    # =========================================================================
    # Inspect every TGZ
    # =========================================================================

    for archive_number, tgz_info in enumerate(
        tgz_infos,
        start=1
    ):

        tgz_start = time.time()

        tgz_name = tgz_info.filename

        print("\n\n")
        print("#" * 110)
        print(
            f"ARCHIVE {archive_number}/{len(tgz_infos)}"
        )
        print("#" * 110)

        print(f"\nOuter-ZIP member:")
        print(tgz_name)

        print(
            f"\nTGZ size: "
            f"{tgz_info.file_size / (1024**2):,.2f} MB"
        )


        # =====================================================================
        # Open TGZ directly from the ZIP as a streaming object.
        #
        # IMPORTANT:
        # No TGZ is written to disk.
        #
        # tarfile mode "r|gz" is sequential streaming mode. This is important
        # for these large archives because Python does not need to load the
        # full TGZ into RAM.
        # =====================================================================

        with outer_zip.open(tgz_info, "r") as tgz_stream:

            with tarfile.open(
                fileobj=tgz_stream,
                mode="r|gz"
            ) as tar:

                member_count = 0
                file_count = 0
                dir_count = 0

                total_uncompressed_bytes = 0

                top_level_counts = Counter()
                extension_counts = Counter()

                final_paths = []
                archive_like_members = []

                member_types = Counter()


                # =============================================================
                # Read TAR metadata sequentially
                # =============================================================

                for member in tar:

                    member_count += 1

                    name = member.name.replace("\\", "/")

                    final_paths.append(
                        (
                            name,
                            member.isdir(),
                            member.size
                        )
                    )

                    # ---------------------------------------------------------
                    # Count top-level branches.
                    # ---------------------------------------------------------

                    clean = name.strip("/")

                    if clean:

                        parts = clean.split("/")

                        top_level_counts[
                            parts[0]
                        ] += 1


                    # ---------------------------------------------------------
                    # Member type
                    # ---------------------------------------------------------

                    if member.isdir():

                        dir_count += 1
                        member_types["directory"] += 1

                    elif member.isfile():

                        file_count += 1
                        member_types["regular_file"] += 1

                        total_uncompressed_bytes += member.size

                        lower = name.lower()

                        if lower.endswith(".tar.gz"):

                            ext = ".tar.gz"

                        else:

                            ext = (
                                Path(lower).suffix
                                or "[no extension]"
                            )

                        extension_counts[ext] += 1

                        if is_archive_like(name):

                            archive_like_members.append(
                                (
                                    name,
                                    member.size
                                )
                            )

                    elif member.issym():

                        member_types["symbolic_link"] += 1

                    elif member.islnk():

                        member_types["hard_link"] += 1

                    else:

                        member_types["other"] += 1


        # =====================================================================
        # Summary for this TGZ
        # =====================================================================

        elapsed = time.time() - tgz_start

        print("\n" + "=" * 110)
        print("FINAL-LAYER SUMMARY")
        print("=" * 110)

        print(
            f"TAR members              : "
            f"{member_count:,}"
        )

        print(
            f"Regular files            : "
            f"{file_count:,}"
        )

        print(
            f"Directories              : "
            f"{dir_count:,}"
        )

        print(
            f"Final uncompressed size  : "
            f"{total_uncompressed_bytes / (1024**3):,.3f} GB"
        )

        print(
            f"Inspection time          : "
            f"{elapsed:,.1f} sec"
        )


        # =====================================================================
        # Top-level branches INSIDE this TGZ
        # =====================================================================

        print("\n" + "-" * 110)
        print("TOP-LEVEL BRANCHES INSIDE THIS TGZ")
        print("-" * 110)

        for branch, count in sorted(
            top_level_counts.items(),
            key=lambda x: x[0].lower()
        ):

            print(
                f"{branch:<80}"
                f"{count:>12,} member(s)"
            )


        # =====================================================================
        # File types
        # =====================================================================

        print("\n" + "-" * 110)
        print("FINAL-LAYER FILE TYPES")
        print("-" * 110)

        for ext, count in sorted(
            extension_counts.items(),
            key=lambda x: (-x[1], x[0])
        ):

            print(
                f"{ext:<25}"
                f"{count:>12,}"
            )


        # =====================================================================
        # Example final paths
        # =====================================================================

        print("\n" + "-" * 110)
        print("EXAMPLE FINAL-LAYER PATHS")
        print("-" * 110)

        for name, is_dir, size in final_paths[:N_SAMPLE_PATHS]:

            if is_dir:

                print(
                    f"[DIR ] {name}"
                )

            else:

                print(
                    f"[FILE] {name}"
                    f"    "
                    f"{size / (1024**2):,.2f} MB"
                )


        # =====================================================================
        # Detect whether there is STILL another compression layer
        # =====================================================================

        print("\n" + "-" * 110)
        print("DEEPER ARCHIVE CHECK")
        print("-" * 110)

        if archive_like_members:

            print(
                "WARNING: archive-like files were found INSIDE this TGZ."
            )

            print(
                "That means this archive may contain another compression layer."
            )

            print(
                f"\nArchive-like members found: "
                f"{len(archive_like_members):,}"
            )

            for name, size in archive_like_members:

                print(
                    f"  {name}"
                    f"    "
                    f"{size / (1024**2):,.2f} MB"
                )

        else:

            print(
                "No additional ZIP/TGZ/TAR/GZ archive layer detected."
            )

            print(
                "This appears to be the final data layer."
            )


        # =====================================================================
        # Complete tree
        # =====================================================================

        if PRINT_COMPLETE_TREE:

            print("\n" + "=" * 110)
            print("COMPLETE FINAL-LAYER TREE")
            print("=" * 110)

            for name, is_dir, size in final_paths:

                print_tree_path(
                    member_name=name,
                    is_dir=is_dir
                )


        # =====================================================================
        # Exact stored paths
        # =====================================================================

        if PRINT_COMPLETE_PATHS:

            print("\n" + "=" * 110)
            print("COMPLETE EXACT PATHS STORED INSIDE TGZ")
            print("=" * 110)

            for i, (name, is_dir, size) in enumerate(
                final_paths,
                start=1
            ):

                kind = "DIR " if is_dir else "FILE"

                print(
                    f"{i:>6}. "
                    f"[{kind}] "
                    f"{name}"
                )


        # Save only metadata summary in RAM.
        all_archive_summaries.append(
            {
                "tgz": tgz_name,
                "members": member_count,
                "files": file_count,
                "directories": dir_count,
                "uncompressed_gb":
                    total_uncompressed_bytes / (1024**3),
                "deeper_archives":
                    len(archive_like_members),
            }
        )


# =============================================================================
# 5. Overall summary across all 18 TGZ archives
# =============================================================================

overall_elapsed = time.time() - overall_start

print("\n\n")
print("=" * 110)
print("OVERALL DEEP INSPECTION SUMMARY")
print("=" * 110)

total_members = sum(
    x["members"]
    for x in all_archive_summaries
)

total_files = sum(
    x["files"]
    for x in all_archive_summaries
)

total_dirs = sum(
    x["directories"]
    for x in all_archive_summaries
)

total_gb = sum(
    x["uncompressed_gb"]
    for x in all_archive_summaries
)

total_deeper = sum(
    x["deeper_archives"]
    for x in all_archive_summaries
)


print(
    f"TGZ archives inspected       : "
    f"{len(all_archive_summaries):,}"
)

print(
    f"Final-layer members          : "
    f"{total_members:,}"
)

print(
    f"Final-layer regular files    : "
    f"{total_files:,}"
)

print(
    f"Final-layer directories      : "
    f"{total_dirs:,}"
)

print(
    f"Total final data size        : "
    f"{total_gb:,.2f} GB"
)

print(
    f"Still-deeper archive members : "
    f"{total_deeper:,}"
)

print(
    f"Total inspection time        : "
    f"{overall_elapsed / 60:,.2f} minutes"
)


# =============================================================================
# 6. Per-archive compact table
# =============================================================================

print("\n" + "-" * 110)
print("PER-ARCHIVE SUMMARY")
print("-" * 110)

print(
    f"{'TGZ':<65}"
    f"{'FILES':>10}"
    f"{'DIRS':>10}"
    f"{'GB':>12}"
    f"{'DEEPER':>10}"
)

print("-" * 110)

for x in all_archive_summaries:

    short_name = Path(x["tgz"]).name

    print(
        f"{short_name:<65}"
        f"{x['files']:>10,}"
        f"{x['directories']:>10,}"
        f"{x['uncompressed_gb']:>12.2f}"
        f"{x['deeper_archives']:>10,}"
    )


# =============================================================================
# 7. Final safety confirmation
# =============================================================================

print("\n" + "=" * 110)
print("READ-ONLY INSPECTION COMPLETE")
print("=" * 110)

print(
    "Nothing was extracted to disk."
)

print(
    "Nothing was deleted."
)

print(
    "Nothing was renamed."
)

print(
    "No output directories were created."
)

print(
    "The TGZ contents were inspected directly as streams from the outer ZIP."
)

print("=" * 110)

DEEP READ-ONLY ARCHIVE INSPECTION
OUTER ZIP -> TGZ -> FINAL TAR CONTENTS

Source ZIP:
C:\Users\rwen13\Desktop\consumer_panel\2004-2020_Panel_Data.zip

Nested TGZ archives found: 18



##############################################################################################################
ARCHIVE 1/18
##############################################################################################################

Outer-ZIP member:
2004-2020_Panel_Data/Consumer_Panel_Data_2004.tgz

TGZ size: 492.62 MB

FINAL-LAYER SUMMARY
TAR members              : 4
Regular files            : 4
Directories              : 0
Final uncompressed size  : 1.820 GB
Inspection time          : 5.0 sec

--------------------------------------------------------------------------------------------------------------
TOP-LEVEL BRANCHES INSIDE THIS TGZ
--------------------------------------------------------------------------------------------------------------
nielsen_extracts                                      

In [2]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import zipfile
import tarfile
import shutil
import os
import threading
import time


# =============================================================================
# Configuration
# =============================================================================

ZIP_PATH = Path(
    r"C:\Users\rwen13\Desktop\consumer_panel\2004-2020_Panel_Data.zip"
)

# Final directory name is EXACTLY the ZIP base name.
# No "_clean", "_extracted", or other suffix is added.
OUTPUT_ROOT = ZIP_PATH.with_suffix("")

# Two parallel TGZ streams are usually a good balance between decompression
# speed and disk-write contention.
#
# If the disk is a fast NVMe SSD, 3 may be worth trying.
# For a normal SSD, keep this at 2.
# For an HDD, use 1.
MAX_WORKERS = 2

# If a final TSV already exists and its byte size exactly matches the TAR
# metadata, treat it as already completed and skip it.
#
# Existing files are NEVER overwritten.
RESUME_IF_SAME_SIZE = True

# Large copy buffer for better throughput.
COPY_BUFFER_BYTES = 16 * 1024 * 1024   # 16 MB


# =============================================================================
# HARD SAFETY RULES
# =============================================================================
#
# This cell:
#
#   DOES:
#     - stream TGZ files directly from the outer ZIP
#     - decompress final TSV files
#     - preserve the exact internal TSV paths
#     - verify extracted file sizes
#     - support safe restart/resume
#
#   DOES NOT:
#     - delete the original ZIP
#     - write the 18 TGZ archives to disk
#     - rename final data files
#     - add "_clean"
#     - add "_extracted"
#     - overwrite existing final files
#     - blindly call extractall()
#
# If anything unexpected occurs, the script stops rather than overwriting data.
# =============================================================================


# =============================================================================
# 1. Validate source archive
# =============================================================================

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Source ZIP does not exist:\n{ZIP_PATH}"
    )

if not ZIP_PATH.is_file():
    raise RuntimeError(
        f"ZIP_PATH is not a file:\n{ZIP_PATH}"
    )

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        f"Source is not a valid ZIP archive:\n{ZIP_PATH}"
    )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. Define the exact structure already confirmed by deep inspection
# =============================================================================
#
# Annual archives:
#
#   Consumer_Panel_Data_YYYY.tgz
#
# contain:
#
#   nielsen_extracts/HMS/YYYY/Annual_Files/panelists_YYYY.tsv
#   nielsen_extracts/HMS/YYYY/Annual_Files/products_extra_YYYY.tsv
#   nielsen_extracts/HMS/YYYY/Annual_Files/purchases_YYYY.tsv
#   nielsen_extracts/HMS/YYYY/Annual_Files/trips_YYYY.tsv
#
# 2020 additionally contains the same three Master files that also appear in:
#
#   Master_Files2004-2020.tgz
#
# To avoid duplicate writes, this cell extracts the Master files ONLY from
# Master_Files2004-2020.tgz.
# =============================================================================

YEARS = list(range(2004, 2021))

ANNUAL_BASENAMES = (
    "panelists_{year}.tsv",
    "products_extra_{year}.tsv",
    "purchases_{year}.tsv",
    "trips_{year}.tsv",
)

MASTER_PATHS = {
    "nielsen_extracts/HMS/Master_Files/Latest/brand_variations.tsv",
    "nielsen_extracts/HMS/Master_Files/Latest/products.tsv",
    "nielsen_extracts/HMS/Master_Files/Latest/retailers.tsv",
}


def expected_annual_paths(year):
    """
    Return the four confirmed annual TSV paths for one panel year.
    """
    base = f"nielsen_extracts/HMS/{year}/Annual_Files"

    return {
        f"{base}/{template.format(year=year)}"
        for template in ANNUAL_BASENAMES
    }


EXPECTED_FINAL_PATHS = set()

for year in YEARS:
    EXPECTED_FINAL_PATHS.update(
        expected_annual_paths(year)
    )

EXPECTED_FINAL_PATHS.update(
    MASTER_PATHS
)

EXPECTED_UNIQUE_FILE_COUNT = len(EXPECTED_FINAL_PATHS)

assert EXPECTED_UNIQUE_FILE_COUNT == 71


# =============================================================================
# 3. Locate the 18 TGZ members in the outer ZIP
# =============================================================================

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    outer_members = {
        info.filename.replace("\\", "/"): info
        for info in zf.infolist()
        if not info.is_dir()
    }


annual_archive_map = {}

for year in YEARS:

    member_name = (
        f"2004-2020_Panel_Data/"
        f"Consumer_Panel_Data_{year}.tgz"
    )

    if member_name not in outer_members:
        raise RuntimeError(
            f"Expected annual TGZ is missing:\n{member_name}"
        )

    annual_archive_map[year] = member_name


MASTER_ARCHIVE_MEMBER = (
    "2004-2020_Panel_Data/"
    "Master_Files2004-2020.tgz"
)

if MASTER_ARCHIVE_MEMBER not in outer_members:
    raise RuntimeError(
        f"Expected Master TGZ is missing:\n"
        f"{MASTER_ARCHIVE_MEMBER}"
    )


print("=" * 90)
print("SAFE STREAMING EXTRACTION")
print("=" * 90)

print(f"\nSource ZIP:")
print(ZIP_PATH)

print(f"\nFinal output root:")
print(OUTPUT_ROOT)

print(
    f"\nConfirmed unique final TSV files: "
    f"{EXPECTED_UNIQUE_FILE_COUNT}"
)

print(
    f"Parallel TGZ workers: "
    f"{MAX_WORKERS}"
)

print(
    "\nOriginal ZIP will be preserved."
)

print("=" * 90)


# =============================================================================
# 4. Safe file-writing helper
# =============================================================================

def safe_write_tar_member(
    tar,
    member,
    output_root,
):
    """
    Extract one regular TAR member safely.

    Extraction procedure:

        TAR stream
            -> temporary .partial file
            -> verify byte size
            -> atomic rename to exact final filename

    Existing final files are never overwritten.
    """

    member_name = member.name.replace("\\", "/").strip("/")

    # -------------------------------------------------------------------------
    # Only expected TSV files may be written.
    # -------------------------------------------------------------------------

    if member_name not in EXPECTED_FINAL_PATHS:
        raise RuntimeError(
            "Attempted extraction of an unexpected TAR member:\n"
            f"{member_name}"
        )

    if not member.isfile():
        raise RuntimeError(
            "Expected final member is not a regular file:\n"
            f"{member_name}"
        )

    relative_path = Path(member_name)

    # -------------------------------------------------------------------------
    # Basic path-traversal protection.
    # -------------------------------------------------------------------------

    if (
        relative_path.is_absolute()
        or ".." in relative_path.parts
    ):
        raise RuntimeError(
            f"Unsafe TAR path detected:\n{member_name}"
        )

    final_path = output_root / relative_path

    final_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # -------------------------------------------------------------------------
    # Safe resume.
    # -------------------------------------------------------------------------

    if final_path.exists():

        existing_size = final_path.stat().st_size

        if (
            RESUME_IF_SAME_SIZE
            and existing_size == member.size
        ):
            return {
                "path": member_name,
                "status": "already_complete",
                "bytes": member.size,
            }

        raise FileExistsError(
            "A final file already exists but cannot be safely reused.\n\n"
            f"File:\n{final_path}\n\n"
            f"Existing size: {existing_size:,} bytes\n"
            f"Expected size: {member.size:,} bytes\n\n"
            "The script will NOT overwrite it."
        )

    # -------------------------------------------------------------------------
    # Temporary partial path.
    #
    # This name exists only during extraction.
    # The final data filename remains unchanged.
    # -------------------------------------------------------------------------

    partial_path = final_path.with_name(
        final_path.name + ".partial"
    )

    # A stale partial file from a previously interrupted extraction is safe
    # to remove because it was never promoted to a verified final TSV.
    if partial_path.exists():
        partial_path.unlink()

    source = tar.extractfile(member)

    if source is None:
        raise RuntimeError(
            f"Could not open TAR member:\n{member_name}"
        )

    try:

        with source, open(
            partial_path,
            "wb",
            buffering=COPY_BUFFER_BYTES,
        ) as destination:

            shutil.copyfileobj(
                source,
                destination,
                length=COPY_BUFFER_BYTES,
            )

        # ---------------------------------------------------------------------
        # Verify exact extracted size.
        # ---------------------------------------------------------------------

        extracted_size = partial_path.stat().st_size

        if extracted_size != member.size:

            partial_path.unlink(
                missing_ok=True
            )

            raise RuntimeError(
                "Extracted file size does not match TAR metadata.\n\n"
                f"File: {member_name}\n"
                f"Expected: {member.size:,} bytes\n"
                f"Actual:   {extracted_size:,} bytes"
            )

        # ---------------------------------------------------------------------
        # Atomic promotion to the final exact filename.
        #
        # os.rename() is used only after confirming final_path does not exist.
        # ---------------------------------------------------------------------

        if final_path.exists():

            partial_path.unlink(
                missing_ok=True
            )

            raise FileExistsError(
                "Final path unexpectedly appeared during extraction:\n"
                f"{final_path}"
            )

        os.rename(
            partial_path,
            final_path
        )

        return {
            "path": member_name,
            "status": "extracted",
            "bytes": member.size,
        }

    except Exception:

        partial_path.unlink(
            missing_ok=True
        )

        raise


# =============================================================================
# 5. Extract one TGZ directly from the outer ZIP
# =============================================================================

def extract_one_tgz(
    archive_member,
    allowed_paths,
    label,
):
    """
    Stream one TGZ directly from the outer ZIP.

    The nested TGZ itself is never written to disk.
    """

    start = time.time()

    extracted_results = []

    seen_regular_files = set()
    unexpected_regular_files = []

    # Each worker opens its own independent ZipFile handle.
    # This is safer for parallel reads.
    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as outer_zip:

        with outer_zip.open(
            archive_member,
            "r",
        ) as tgz_stream:

            # Streaming gzip+tar mode.
            #
            # This avoids:
            #   ZIP -> TGZ file -> TAR file -> TSV
            #
            # and instead performs:
            #
            #   ZIP -> TGZ stream -> TSV
            #
            with tarfile.open(
                fileobj=tgz_stream,
                mode="r|gz",
            ) as tar:

                for member in tar:

                    member_name = (
                        member.name
                        .replace("\\", "/")
                        .strip("/")
                    )

                    if not member.isfile():
                        continue

                    seen_regular_files.add(
                        member_name
                    )

                    # ---------------------------------------------------------
                    # Only extract paths assigned to this TGZ.
                    #
                    # Example:
                    # 2020 TGZ contains duplicate Master files, but this task
                    # extracts only the four 2020 annual files.
                    # ---------------------------------------------------------

                    if member_name in allowed_paths:

                        result = safe_write_tar_member(
                            tar=tar,
                            member=member,
                            output_root=OUTPUT_ROOT,
                        )

                        extracted_results.append(
                            result
                        )

                    else:

                        # Known duplicate Master paths inside the 2020 TGZ are
                        # deliberately ignored.
                        if (
                            archive_member.endswith(
                                "Consumer_Panel_Data_2020.tgz"
                            )
                            and member_name in MASTER_PATHS
                        ):
                            continue

                        unexpected_regular_files.append(
                            member_name
                        )

    # -------------------------------------------------------------------------
    # Validate archive structure.
    # -------------------------------------------------------------------------

    missing = (
        set(allowed_paths)
        - seen_regular_files
    )

    if missing:
        raise RuntimeError(
            f"{label}: expected files were missing:\n"
            + "\n".join(
                f"  {x}"
                for x in sorted(missing)
            )
        )

    if unexpected_regular_files:
        raise RuntimeError(
            f"{label}: unexpected file(s) detected:\n"
            + "\n".join(
                f"  {x}"
                for x in sorted(
                    unexpected_regular_files
                )
            )
        )

    elapsed = time.time() - start

    return {
        "label": label,
        "archive_member": archive_member,
        "results": extracted_results,
        "elapsed_seconds": elapsed,
    }


# =============================================================================
# 6. Build extraction tasks
# =============================================================================

tasks = []

for year in YEARS:

    tasks.append(
        {
            "archive_member":
                annual_archive_map[year],

            "allowed_paths":
                expected_annual_paths(year),

            "label":
                f"Annual {year}",
        }
    )


# Extract Master files ONLY from the dedicated Master archive.
tasks.append(
    {
        "archive_member":
            MASTER_ARCHIVE_MEMBER,

        "allowed_paths":
            MASTER_PATHS,

        "label":
            "Master Files",
    }
)


# =============================================================================
# 7. Run extraction
# =============================================================================

overall_start = time.time()

completed_results = []
failure = None

print("\nStarting extraction...\n")


with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    future_map = {
        executor.submit(
            extract_one_tgz,
            task["archive_member"],
            task["allowed_paths"],
            task["label"],
        ): task
        for task in tasks
    }

    for future in as_completed(future_map):

        task = future_map[future]

        try:

            result = future.result()

            completed_results.append(
                result
            )

            n_extracted = sum(
                x["status"] == "extracted"
                for x in result["results"]
            )

            n_existing = sum(
                x["status"] == "already_complete"
                for x in result["results"]
            )

            print(
                f"[OK] {result['label']:<18} | "
                f"new={n_extracted:<2} | "
                f"existing={n_existing:<2} | "
                f"{result['elapsed_seconds']:>6.1f} sec"
            )

        except Exception as exc:

            failure = (
                task,
                exc,
            )

            print(
                f"\n[FAILED] {task['label']}"
            )

            print(exc)

            # Stop scheduling useful work as soon as possible.
            for other_future in future_map:
                other_future.cancel()

            break


# =============================================================================
# 8. Stop immediately if an extraction task failed
# =============================================================================

if failure is not None:

    task, exc = failure

    raise RuntimeError(
        "\nExtraction stopped safely.\n"
        "The original ZIP remains untouched.\n"
        "Verified final TSV files already completed were preserved.\n"
        "No existing final TSV was overwritten."
    ) from exc


# =============================================================================
# 9. Final verification of all 71 UNIQUE final TSV files
# =============================================================================

print("\n" + "=" * 90)
print("FINAL VERIFICATION")
print("=" * 90)

missing_final = []
wrong_size_final = []
partial_files = []


# Build expected sizes directly from the actual TAR metadata results produced
# during extraction.
expected_sizes = {}

for archive_result in completed_results:

    for file_result in archive_result["results"]:

        expected_sizes[
            file_result["path"]
        ] = file_result["bytes"]


for relative_name in sorted(
    EXPECTED_FINAL_PATHS
):

    final_path = (
        OUTPUT_ROOT
        / Path(relative_name)
    )

    if not final_path.exists():

        missing_final.append(
            relative_name
        )

        continue

    expected_size = expected_sizes.get(
        relative_name
    )

    if expected_size is None:

        wrong_size_final.append(
            (
                relative_name,
                "No expected size recorded",
            )
        )

        continue

    actual_size = final_path.stat().st_size

    if actual_size != expected_size:

        wrong_size_final.append(
            (
                relative_name,
                (
                    f"expected={expected_size:,}, "
                    f"actual={actual_size:,}"
                ),
            )
        )


# Search only for our own temporary .partial files.
for current_root, _, filenames in os.walk(
    OUTPUT_ROOT
):

    current_root = Path(current_root)

    for filename in filenames:

        if filename.endswith(".partial"):

            partial_files.append(
                current_root / filename
            )


# =============================================================================
# 10. Report final status
# =============================================================================

elapsed_total = (
    time.time()
    - overall_start
)

verified_count = (
    EXPECTED_UNIQUE_FILE_COUNT
    - len(missing_final)
    - len(wrong_size_final)
)


print(
    f"Expected unique TSV files : "
    f"{EXPECTED_UNIQUE_FILE_COUNT}"
)

print(
    f"Verified TSV files        : "
    f"{verified_count}"
)

print(
    f"Missing TSV files         : "
    f"{len(missing_final)}"
)

print(
    f"Wrong-size TSV files      : "
    f"{len(wrong_size_final)}"
)

print(
    f"Remaining .partial files  : "
    f"{len(partial_files)}"
)

print(
    f"Total elapsed time        : "
    f"{elapsed_total / 60:,.2f} minutes"
)


if missing_final:

    print("\nMissing files:")

    for x in missing_final:
        print(f"  {x}")


if wrong_size_final:

    print("\nWrong-size files:")

    for name, reason in wrong_size_final:
        print(f"  {name}: {reason}")


if partial_files:

    print("\nPartial files:")

    for x in partial_files:
        print(f"  {x}")


# =============================================================================
# 11. Final success / failure
# =============================================================================

print("\n" + "=" * 90)

if (
    not missing_final
    and not wrong_size_final
    and not partial_files
):

    print("EXTRACTION COMPLETE AND VERIFIED.")

    print(
        "\nAll 71 unique final TSV files were extracted successfully."
    )

    print(
        "\nFinal paths and filenames exactly match the archive structure."
    )

    print(
        "\nThe duplicate Master files embedded in the 2020 TGZ were "
        "intentionally skipped; the authoritative copies were extracted "
        "from Master_Files2004-2020.tgz."
    )

    print(
        "\nThe original source ZIP was NOT deleted or modified:"
    )

    print(ZIP_PATH)

    print(
        "\nFinal data root:"
    )

    print(OUTPUT_ROOT)

else:

    raise RuntimeError(
        "Final verification failed. "
        "The source ZIP remains untouched."
    )

print("=" * 90)

SAFE STREAMING EXTRACTION

Source ZIP:
C:\Users\rwen13\Desktop\consumer_panel\2004-2020_Panel_Data.zip

Final output root:
C:\Users\rwen13\Desktop\consumer_panel\2004-2020_Panel_Data

Confirmed unique final TSV files: 71
Parallel TGZ workers: 2

Original ZIP will be preserved.

Starting extraction...

[OK] Annual 2004        | new=4  | existing=0  |   13.7 sec
[OK] Annual 2005        | new=4  | existing=0  |   13.8 sec
[OK] Annual 2006        | new=4  | existing=0  |   12.2 sec
[OK] Annual 2007        | new=4  | existing=0  |   21.1 sec
[OK] Annual 2008        | new=4  | existing=0  |   20.3 sec
[OK] Annual 2009        | new=4  | existing=0  |   20.3 sec
[OK] Annual 2010        | new=4  | existing=0  |   19.6 sec
[OK] Annual 2011        | new=4  | existing=0  |   23.3 sec
[OK] Annual 2012        | new=4  | existing=0  |   21.0 sec
[OK] Annual 2013        | new=4  | existing=0  |   20.1 sec
[OK] Annual 2014        | new=4  | existing=0  |   22.9 sec
[OK] Annual 2015        | new=4  | ex

In [6]:
# =============================================================================
# MILK RETAIL DATA — STAGE 1: READ-ONLY RECURSIVE ARCHIVE INSPECTION
# =============================================================================
#
# IMPORTANT
# ---------
# This cell ONLY inspects archive structure.
#
# It does NOT:
#   - extract final data files
#   - create an extraction folder
#   - delete anything
#   - rename anything
#   - overwrite anything
#   - modify the original archive
#
# Archive:
#   C:\Users\rwen13\Desktop\retail\Milk_2006-2020.tgz
#
# After running this cell, paste the COMPLETE printed output back to ChatGPT.
# We will then write a separate extraction cell based on the real structure.
#
# =============================================================================

from __future__ import annotations

import bz2
import gzip
import lzma
import os
import tarfile
import tempfile
import zipfile

from collections import Counter
from pathlib import Path, PurePosixPath


# =============================================================================
# 1. INPUT ARCHIVE
# =============================================================================

ARCHIVE_PATH = Path(
    r"C:\Users\rwen13\Desktop\retail\Milk_2006-2020.tgz"
)

# Safety limit for recursive nested archives.
MAX_DEPTH = 15

# Streaming buffer for temporary nested-archive inspection.
COPY_BUFFER_BYTES = 8 * 1024 * 1024  # 8 MB


# =============================================================================
# 2. HELPERS
# =============================================================================

def normalize_member_name(name: str) -> str:
    """Normalize archive-internal paths for display only."""
    return name.replace("\\", "/").lstrip("/")


def human_bytes(n: int | None) -> str:
    """Convert byte count to readable size."""
    if n is None:
        return "unknown"

    size = float(n)

    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024 or unit == "TB":
            if unit == "B":
                return f"{int(size):,} {unit}"
            return f"{size:,.2f} {unit}"
        size /= 1024

    return f"{n:,} B"


def detect_archive_type(name: str) -> str | None:
    """
    Detect supported archive/compression type from filename.
    """
    x = name.lower()

    if x.endswith(".zip"):
        return "zip"

    if x.endswith((".tar.gz", ".tgz")):
        return "tar.gz"

    if x.endswith((".tar.bz2", ".tbz", ".tbz2")):
        return "tar.bz2"

    if x.endswith((".tar.xz", ".txz")):
        return "tar.xz"

    if x.endswith(".tar"):
        return "tar"

    if x.endswith(".gz"):
        return "gzip"

    if x.endswith(".bz2"):
        return "bzip2"

    if x.endswith(".xz"):
        return "xz"

    return None


def infer_file_type(name: str) -> str:
    """Infer a simple final file type for reporting."""
    x = name.lower()

    mappings = [
        (".tar.gz", "TAR.GZ archive"),
        (".tar.bz2", "TAR.BZ2 archive"),
        (".tar.xz", "TAR.XZ archive"),
        (".tgz", "TGZ archive"),
        (".zip", "ZIP archive"),
        (".tar", "TAR archive"),
        (".gz", "GZIP-compressed file"),
        (".bz2", "BZIP2-compressed file"),
        (".xz", "XZ-compressed file"),
        (".tsv", "TSV"),
        (".csv", "CSV"),
        (".txt", "Text"),
        (".dat", "DAT"),
        (".parquet", "Parquet"),
        (".sas7bdat", "SAS dataset"),
        (".dta", "Stata dataset"),
        (".xlsx", "Excel workbook"),
        (".xls", "Excel workbook"),
        (".json", "JSON"),
        (".xml", "XML"),
    ]

    for suffix, label in mappings:
        if x.endswith(suffix):
            return label

    suffix = Path(name).suffix

    if suffix:
        return f"{suffix[1:].upper()} file"

    return "No extension / unknown"


# =============================================================================
# 3. RESULT CONTAINERS
# =============================================================================

archive_layers = []
final_files = []
errors = []


def add_archive(path, archive_type, size, depth):
    archive_layers.append(
        {
            "path": path,
            "archive_type": archive_type,
            "size": size,
            "depth": depth,
        }
    )


def add_final(path, size, depth):
    final_files.append(
        {
            "path": path,
            "size": size,
            "depth": depth,
            "file_type": infer_file_type(path),
        }
    )


def add_error(path, message):
    errors.append(
        {
            "path": path,
            "error": message,
        }
    )


# =============================================================================
# 4. STREAM NESTED ARCHIVE TO TEMPORARY SEEKABLE FILE
# =============================================================================

def stream_to_temp(source_stream, suffix=""):
    """
    Stream a nested archive member to a temporary file.

    This is ONLY for inspection when random access is required.
    The temporary file is deleted immediately after inspection.
    """
    tmp = tempfile.NamedTemporaryFile(
        mode="wb",
        suffix=suffix,
        delete=False,
    )

    tmp_path = tmp.name

    try:
        with tmp:
            while True:
                block = source_stream.read(COPY_BUFFER_BYTES)

                if not block:
                    break

                tmp.write(block)

        return tmp_path

    except Exception:
        try:
            os.remove(tmp_path)
        except OSError:
            pass
        raise


# =============================================================================
# 5. ZIP INSPECTION
# =============================================================================

def inspect_zip(path, virtual_path, depth):

    with zipfile.ZipFile(path, "r") as zf:

        for info in zf.infolist():

            if info.is_dir():
                continue

            member = normalize_member_name(info.filename)

            child_virtual = (
                f"{virtual_path} :: {member}"
            )

            nested_type = detect_archive_type(member)

            if nested_type is None:

                add_final(
                    child_virtual,
                    info.file_size,
                    depth + 1,
                )

                continue

            suffix = "".join(
                Path(member).suffixes
            )

            with zf.open(info, "r") as src:
                tmp_path = stream_to_temp(
                    src,
                    suffix=suffix,
                )

            try:

                inspect_archive(
                    tmp_path,
                    child_virtual,
                    depth + 1,
                )

            finally:

                try:
                    os.remove(tmp_path)
                except OSError:
                    pass


# =============================================================================
# 6. TAR / TGZ INSPECTION
# =============================================================================

def inspect_tar(path, virtual_path, depth):

    with tarfile.open(path, mode="r:*") as tf:

        for member in tf:

            if not member.isfile():
                continue

            member_name = normalize_member_name(
                member.name
            )

            child_virtual = (
                f"{virtual_path} :: {member_name}"
            )

            nested_type = detect_archive_type(
                member_name
            )

            if nested_type is None:

                add_final(
                    child_virtual,
                    member.size,
                    depth + 1,
                )

                continue

            source = tf.extractfile(member)

            if source is None:

                add_error(
                    child_virtual,
                    "Could not open TAR member stream.",
                )

                continue

            suffix = "".join(
                Path(member_name).suffixes
            )

            try:

                tmp_path = stream_to_temp(
                    source,
                    suffix=suffix,
                )

            finally:
                source.close()

            try:

                inspect_archive(
                    tmp_path,
                    child_virtual,
                    depth + 1,
                )

            finally:

                try:
                    os.remove(tmp_path)
                except OSError:
                    pass


# =============================================================================
# 7. SINGLE-FILE COMPRESSION INSPECTION
# =============================================================================

def inspect_single_compressed(
    path,
    virtual_path,
    depth,
    archive_type,
):

    path = Path(path)

    if archive_type == "gzip":
        opener = gzip.open
        child_name = path.name[:-3]

    elif archive_type == "bzip2":
        opener = bz2.open
        child_name = path.name[:-4]

    elif archive_type == "xz":
        opener = lzma.open
        child_name = path.name[:-3]

    else:
        raise ValueError(
            f"Unsupported compression type: {archive_type}"
        )

    child_virtual = (
        f"{virtual_path} :: {child_name}"
    )

    child_type = detect_archive_type(
        child_name
    )

    # If decompression produces a normal file,
    # record it without writing the full decompressed data.
    if child_type is None:

        add_final(
            child_virtual,
            None,
            depth + 1,
        )

        return

    # If decompression reveals another archive,
    # stream it temporarily and inspect recursively.
    with opener(path, "rb") as src:

        suffix = "".join(
            Path(child_name).suffixes
        )

        tmp_path = stream_to_temp(
            src,
            suffix=suffix,
        )

    try:

        inspect_archive(
            tmp_path,
            child_virtual,
            depth + 1,
        )

    finally:

        try:
            os.remove(tmp_path)
        except OSError:
            pass


# =============================================================================
# 8. GENERAL RECURSIVE INSPECTION
# =============================================================================

def inspect_archive(path, virtual_path, depth=0):

    if depth > MAX_DEPTH:

        add_error(
            virtual_path,
            f"Maximum recursion depth {MAX_DEPTH} exceeded.",
        )

        return

    path = Path(path)

    archive_type = detect_archive_type(
        virtual_path
    )

    # Temporary nested files may have unusual names.
    # Fall back to the actual temporary filename if required.
    if archive_type is None:
        archive_type = detect_archive_type(
            path.name
        )

    if archive_type is None:

        add_final(
            virtual_path,
            path.stat().st_size,
            depth,
        )

        return

    try:
        size = path.stat().st_size
    except OSError:
        size = None

    add_archive(
        virtual_path,
        archive_type,
        size,
        depth,
    )

    try:

        if archive_type == "zip":

            inspect_zip(
                path,
                virtual_path,
                depth,
            )

        elif archive_type in {
            "tar",
            "tar.gz",
            "tar.bz2",
            "tar.xz",
        }:

            inspect_tar(
                path,
                virtual_path,
                depth,
            )

        elif archive_type in {
            "gzip",
            "bzip2",
            "xz",
        }:

            inspect_single_compressed(
                path,
                virtual_path,
                depth,
                archive_type,
            )

        else:

            add_error(
                virtual_path,
                f"Unsupported archive type: {archive_type}",
            )

    except Exception as exc:

        add_error(
            virtual_path,
            f"{type(exc).__name__}: {exc}",
        )


# =============================================================================
# 9. VALIDATE INPUT
# =============================================================================

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        f"Archive does not exist:\n{ARCHIVE_PATH}"
    )

if not ARCHIVE_PATH.is_file():
    raise ValueError(
        f"Path is not a file:\n{ARCHIVE_PATH}"
    )


# =============================================================================
# 10. RUN
# =============================================================================

print("=" * 110)
print("MILK RETAIL DATA — READ-ONLY RECURSIVE INSPECTION")
print("=" * 110)

print()
print(f"Archive path : {ARCHIVE_PATH}")
print(f"Archive size : {human_bytes(ARCHIVE_PATH.stat().st_size)}")
print(f"Archive type : {detect_archive_type(ARCHIVE_PATH.name)}")

print()
print("Scanning archive layers recursively...")
print()

inspect_archive(
    ARCHIVE_PATH,
    ARCHIVE_PATH.name,
    depth=0,
)


# =============================================================================
# 11. TOP-LEVEL BRANCHES
# =============================================================================

def get_top_branch(path_string):

    if " :: " not in path_string:
        return None

    inner = path_string.split(
        " :: ",
        1,
    )[1]

    inner = inner.split(
        " :: ",
        1,
    )[0]

    parts = PurePosixPath(
        normalize_member_name(inner)
    ).parts

    if not parts:
        return "(root)"

    return parts[0]


all_detected_paths = (
    [x["path"] for x in archive_layers]
    + [x["path"] for x in final_files]
)

top_branches = sorted(
    {
        branch
        for path in all_detected_paths
        if (branch := get_top_branch(path)) is not None
    }
)


print("=" * 110)
print("TOP-LEVEL BRANCHES")
print("=" * 110)

print(f"Detected top-level branches: {len(top_branches):,}")
print()

for i, branch in enumerate(top_branches, 1):
    print(f"{i:>4}. {branch}")


# =============================================================================
# 12. ALL ARCHIVE LAYERS
# =============================================================================

print()
print("=" * 110)
print("ALL ARCHIVE / COMPRESSION LAYERS")
print("=" * 110)

print(f"Detected archive layers: {len(archive_layers):,}")
print()

for i, item in enumerate(archive_layers, 1):

    print(f"{i:>4}. {item['path']}")
    print(f"      type  : {item['archive_type']}")
    print(f"      size  : {human_bytes(item['size'])}")
    print(f"      depth : {item['depth']}")
    print()


# =============================================================================
# 13. NESTED ARCHIVES
# =============================================================================

nested_archives = [
    x
    for x in archive_layers
    if x["depth"] > 0
]

print()
print("=" * 110)
print("NESTED ARCHIVE CANDIDATES")
print("=" * 110)

print(f"Detected nested/archive-like files: {len(nested_archives):,}")
print()

for i, item in enumerate(nested_archives, 1):

    print(f"{i:>4}. {item['path']}")
    print(f"      archive type : {item['archive_type']}")
    print(f"      stored size  : {human_bytes(item['size'])}")
    print(f"      depth        : {item['depth']}")
    print()


# =============================================================================
# 14. FINAL-LAYER FILES
# =============================================================================

print()
print("=" * 110)
print("FINAL-LAYER FILES")
print("=" * 110)

print(f"Detected final-layer files: {len(final_files):,}")
print()

for i, item in enumerate(final_files, 1):

    compression_left = (
        detect_archive_type(item["path"]) is not None
    )

    print(f"{i:>4}. {item['path']}")
    print(f"      file type        : {item['file_type']}")
    print(f"      file size        : {human_bytes(item['size'])}")
    print(f"      depth            : {item['depth']}")
    print(
        "      compression left : "
        f"{'YES' if compression_left else 'NO'}"
    )
    print()


# =============================================================================
# 15. FINAL FILE TYPE SUMMARY
# =============================================================================

print()
print("=" * 110)
print("FINAL FILE-TYPE SUMMARY")
print("=" * 110)

counts = Counter(
    x["file_type"]
    for x in final_files
)

for file_type, count in sorted(
    counts.items(),
    key=lambda z: (-z[1], z[0]),
):
    print(
        f"{file_type:<40} {count:>10,}"
    )


# =============================================================================
# 16. ERROR / WARNING REPORT
# =============================================================================

print()
print("=" * 110)
print("INSPECTION ERRORS / WARNINGS")
print("=" * 110)

if errors:

    print(f"Count: {len(errors):,}")
    print()

    for i, item in enumerate(errors, 1):

        print(f"{i:>4}. {item['path']}")
        print(f"      {item['error']}")
        print()

else:

    print("None.")


# =============================================================================
# 17. FINAL SUMMARY
# =============================================================================

max_depth = max(
    [
        x["depth"]
        for x in archive_layers + final_files
    ],
    default=0,
)

print()
print("=" * 110)
print("INSPECTION SUMMARY")
print("=" * 110)

print(f"Top-level branches      : {len(top_branches):,}")
print(f"Archive layers detected : {len(archive_layers):,}")
print(f"Nested archive files    : {len(nested_archives):,}")
print(f"Final-layer files       : {len(final_files):,}")
print(f"Maximum archive depth   : {max_depth:,}")
print(f"Inspection errors       : {len(errors):,}")

print()
print("READ-ONLY INSPECTION COMPLETE.")
print("No extraction directory was created.")
print("No final data files were extracted.")
print("The original archive was not modified.")
print()
print("Paste the COMPLETE output back to ChatGPT before extraction.")

MILK RETAIL DATA — READ-ONLY RECURSIVE INSPECTION

Archive path : C:\Users\rwen13\Desktop\retail\Milk_2006-2020.tgz
Archive size : 11.64 GB
Archive type : tar.gz

Scanning archive layers recursively...

TOP-LEVEL BRANCHES
Detected top-level branches: 1

   1. nielsen_extracts

ALL ARCHIVE / COMPRESSION LAYERS
Detected archive layers: 1

   1. Milk_2006-2020.tgz
      type  : tar.gz
      size  : 11.64 GB
      depth : 0


NESTED ARCHIVE CANDIDATES
Detected nested/archive-like files: 0


FINAL-LAYER FILES
Detected final-layer files: 159

   1. Milk_2006-2020.tgz :: nielsen_extracts/RMS/2020/Movement_Files/2506_2020/1633_2020.tsv
      file type        : TSV
      file size        : 53.82 MB
      depth            : 1
      compression left : NO

   2. Milk_2006-2020.tgz :: nielsen_extracts/RMS/2020/Movement_Files/2506_2020/1716_2020.tsv
      file type        : TSV
      file size        : 7.98 KB
      depth            : 1
      compression left : NO

   3. Milk_2006-2020.tgz :: nielse

In [7]:
# =============================================================================
# MILK RETAIL DATA — SAFE STREAMING EXTRACTION
# =============================================================================
#
# Archive:
#   C:\Users\rwen13\Desktop\retail\Milk_2006-2020.tgz
#
# Output:
#   C:\Users\rwen13\Desktop\retail\nielsen_extracts\...
#
# Safety rules:
#   - Preserve all original paths and filenames exactly.
#   - Never overwrite an existing final file.
#   - Skip an existing file only when its size matches the TAR metadata.
#   - Write each new file as *.partial first.
#   - Verify file size before promoting *.partial to the final filename.
#   - Never delete or modify the original archive.
#   - Stream data instead of loading large files into memory.
#   - Use at most 4 CPU cores.
#
# =============================================================================

from pathlib import Path, PurePosixPath
import os
import shutil
import subprocess
import tarfile


# =============================================================================
# 1. SETTINGS
# =============================================================================

ARCHIVE_PATH = Path(
    r"C:\Users\rwen13\Desktop\retail\Milk_2006-2020.tgz"
)

# Extract directly beside the archive.
# The archive itself already contains the "nielsen_extracts" directory.
OUTPUT_ROOT = ARCHIVE_PATH.parent

MAX_CORES = 4
EXPECTED_FILES = 159

COPY_BUFFER = 16 * 1024 * 1024  # 16 MB


# =============================================================================
# 2. BASIC VALIDATION
# =============================================================================

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(
        f"Archive not found:\n{ARCHIVE_PATH}"
    )

print("=" * 100)
print("MILK RETAIL DATA — SAFE EXTRACTION")
print("=" * 100)

print(f"Archive     : {ARCHIVE_PATH}")
print(f"Output root : {OUTPUT_ROOT}")
print(f"CPU limit   : {MAX_CORES}")
print()


# =============================================================================
# 3. SAFE PATH CHECK
# =============================================================================

def safe_output_path(member_name: str) -> Path:
    """
    Convert a TAR member path into a safe local output path.

    Reject absolute paths and '..' traversal.
    """
    member_name = member_name.replace("\\", "/")
    rel = PurePosixPath(member_name)

    if rel.is_absolute() or ".." in rel.parts:
        raise RuntimeError(
            f"Unsafe archive path detected: {member_name}"
        )

    return OUTPUT_ROOT.joinpath(*rel.parts)


# =============================================================================
# 4. OPEN TAR STREAM
# =============================================================================

# pigz provides multithreaded gzip decompression.
# If available, use exactly 4 worker threads.
pigz_path = shutil.which("pigz")

pigz_process = None

if pigz_path:

    print(f"Decompressor : pigz ({MAX_CORES} cores)")
    print()

    pigz_process = subprocess.Popen(
        [
            pigz_path,
            "-p",
            str(MAX_CORES),
            "-dc",
            str(ARCHIVE_PATH),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=0,
    )

    if pigz_process.stdout is None:
        raise RuntimeError("Could not open pigz output stream.")

    tar = tarfile.open(
        fileobj=pigz_process.stdout,
        mode="r|",
    )

else:

    print("Decompressor : Python gzip streaming")
    print(
        "Note         : pigz was not found, so decompression "
        "will use the standard single-stream gzip reader."
    )
    print()

    tar = tarfile.open(
        ARCHIVE_PATH,
        mode="r|gz",
    )


# =============================================================================
# 5. STREAMING EXTRACTION
# =============================================================================

n_seen = 0
n_extracted = 0
n_skipped = 0
bytes_written = 0


try:

    with tar:

        for member in tar:

            # Directories are created automatically when their files arrive.
            if not member.isfile():
                continue

            n_seen += 1

            final_path = safe_output_path(member.name)

            partial_path = final_path.with_name(
                final_path.name + ".partial"
            )

            expected_size = int(member.size)


            # -----------------------------------------------------------------
            # Safe resume
            # -----------------------------------------------------------------

            if final_path.exists():

                actual_size = final_path.stat().st_size

                if actual_size == expected_size:

                    n_skipped += 1

                    print(
                        f"[{n_seen:>3}/{EXPECTED_FILES}] SKIP  "
                        f"{member.name}"
                    )

                    continue

                raise FileExistsError(
                    "\nExisting final file has the WRONG size.\n"
                    "It will NOT be overwritten.\n\n"
                    f"File          : {final_path}\n"
                    f"Expected size : {expected_size:,} bytes\n"
                    f"Actual size   : {actual_size:,} bytes"
                )


            # -----------------------------------------------------------------
            # Prepare destination
            # -----------------------------------------------------------------

            final_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            # A stale partial file is not considered a final extracted file.
            # Remove it before restarting this member.
            if partial_path.exists():
                partial_path.unlink()


            # -----------------------------------------------------------------
            # Stream member -> .partial
            # -----------------------------------------------------------------

            source = tar.extractfile(member)

            if source is None:
                raise RuntimeError(
                    f"Could not open archive member: {member.name}"
                )

            print(
                f"[{n_seen:>3}/{EXPECTED_FILES}] WRITE "
                f"{member.name}"
            )

            try:

                with source, open(
                    partial_path,
                    "wb",
                    buffering=COPY_BUFFER,
                ) as destination:

                    shutil.copyfileobj(
                        source,
                        destination,
                        length=COPY_BUFFER,
                    )

            except Exception:
                # Keep the .partial file visible for diagnostics.
                raise


            # -----------------------------------------------------------------
            # Verify exact byte size
            # -----------------------------------------------------------------

            written_size = partial_path.stat().st_size

            if written_size != expected_size:

                raise IOError(
                    "\nSize verification failed.\n"
                    "The partial file was NOT promoted.\n\n"
                    f"File          : {member.name}\n"
                    f"Expected size : {expected_size:,} bytes\n"
                    f"Written size  : {written_size:,} bytes\n"
                    f"Partial file  : {partial_path}"
                )


            # -----------------------------------------------------------------
            # Promote .partial -> final
            # -----------------------------------------------------------------

            # Check again immediately before promotion.
            if final_path.exists():
                raise FileExistsError(
                    f"Final path appeared during extraction; "
                    f"refusing to overwrite:\n{final_path}"
                )

            # On Windows, rename refuses to overwrite an existing destination.
            partial_path.rename(final_path)

            n_extracted += 1
            bytes_written += written_size


finally:

    # -------------------------------------------------------------------------
    # Validate pigz completion if pigz was used.
    # -------------------------------------------------------------------------

    if pigz_process is not None:

        if pigz_process.stdout is not None:
            pigz_process.stdout.close()

        return_code = pigz_process.wait()

        stderr_text = ""

        if pigz_process.stderr is not None:
            stderr_text = pigz_process.stderr.read().decode(
                errors="replace"
            )

        if return_code != 0:
            raise RuntimeError(
                f"pigz failed with return code {return_code}.\n"
                f"{stderr_text}"
            )


# =============================================================================
# 6. FINAL SUMMARY
# =============================================================================

print()
print("=" * 100)
print("EXTRACTION COMPLETE")
print("=" * 100)

print(f"Archive files encountered : {n_seen:,}")
print(f"New files extracted       : {n_extracted:,}")
print(f"Existing files skipped    : {n_skipped:,}")
print(
    f"New data written          : "
    f"{bytes_written / (1024**3):,.2f} GB"
)

if n_seen != EXPECTED_FILES:
    print()
    print(
        f"WARNING: inspection reported {EXPECTED_FILES} files, "
        f"but extraction encountered {n_seen}."
    )
else:
    print()
    print(
        f"Verified file count       : {EXPECTED_FILES}"
    )

print()
print("Original archive preserved:")
print(ARCHIVE_PATH)

print()
print("Output:")
print(OUTPUT_ROOT / "nielsen_extracts")

print()
print("DONE.")

MILK RETAIL DATA — SAFE EXTRACTION
Archive     : C:\Users\rwen13\Desktop\retail\Milk_2006-2020.tgz
Output root : C:\Users\rwen13\Desktop\retail
CPU limit   : 4

Decompressor : Python gzip streaming
Note         : pigz was not found, so decompression will use the standard single-stream gzip reader.

[  1/159] WRITE nielsen_extracts/RMS/2020/Movement_Files/2506_2020/1633_2020.tsv
[  2/159] WRITE nielsen_extracts/RMS/2020/Movement_Files/2506_2020/1716_2020.tsv
[  3/159] WRITE nielsen_extracts/RMS/2013/Movement_Files/2506_2013/3592_2013.tsv
[  4/159] WRITE nielsen_extracts/RMS/2020/Movement_Files/2506_2020/3592_2020.tsv
[  5/159] WRITE nielsen_extracts/RMS/2015/Movement_Files/2506_2015/3592_2015.tsv
[  6/159] WRITE nielsen_extracts/RMS/2006/Movement_Files/2506_2006/3592_2006.tsv
[  7/159] WRITE nielsen_extracts/RMS/2010/Movement_Files/2506_2010/3592_2010.tsv
[  8/159] WRITE nielsen_extracts/RMS/2007/Movement_Files/2506_2007/3592_2007.tsv
[  9/159] WRITE nielsen_extracts/RMS/2017/Movement_F

In [2]:
# =============================================================================
# Inspect 2008 store file
# Goal:
#   1. Read the store file safely.
#   2. Print all column names.
#   3. Print the first 3 rows.
#   4. Identify columns that may describe store type / channel / format.
# =============================================================================

import pandas as pd

STORE_PATH = (
    r"C:\Users\rwen13\Desktop\retail_milk"
    r"\nielsen_extracts\RMS\2008\Annual_Files\stores_2008.tsv"
)

# Read only the first few rows.
# This avoids loading the full store file into memory.
stores_sample = pd.read_csv(
    STORE_PATH,
    sep="\t",
    nrows=3,
    dtype=str
)

print("=" * 100)
print("COLUMN NAMES")
print("=" * 100)

for i, col in enumerate(stores_sample.columns, start=1):
    print(f"{i:>3}. {col}")


print()
print("=" * 100)
print("FIRST 3 ROWS")
print("=" * 100)

print(
    stores_sample.to_string(
        index=False
    )
)


# -----------------------------------------------------------------------------
# Search for columns that may contain store-type information
# -----------------------------------------------------------------------------

keywords = [
    "store",
    "type",
    "channel",
    "format",
    "retail",
    "market",
    "chain",
]

candidate_columns = [
    col
    for col in stores_sample.columns
    if any(
        keyword in col.lower()
        for keyword in keywords
    )
]

print()
print("=" * 100)
print("POSSIBLE STORE-TYPE COLUMNS")
print("=" * 100)

if candidate_columns:

    for col in candidate_columns:
        print(f"- {col}")

    print()
    print("Values in the first 3 rows:")
    print()

    print(
        stores_sample[
            candidate_columns
        ].to_string(index=False)
    )

else:
    print("No obvious store-type column was detected from column names.")

COLUMN NAMES
  1. store_code_uc
  2. year
  3. parent_code
  4. retailer_code
  5. channel_code
  6. store_zip3
  7. fips_state_code
  8. fips_state_descr
  9. fips_county_code
 10. fips_county_descr
 11. dma_code
 12. dma_descr

FIRST 3 ROWS
store_code_uc year parent_code retailer_code channel_code store_zip3 fips_state_code fips_state_descr fips_county_code fips_county_descr dma_code              dma_descr
        12493 2008        8106           NaN            C        010              25               MA               13           HAMPDEN      543 SPRINGFIELD-HOLYOKE MA
        23374 2008        8106           NaN            C        010              25               MA               13           HAMPDEN      543 SPRINGFIELD-HOLYOKE MA
        24180 2008        8106           NaN            C        010              25               MA               13           HAMPDEN      543 SPRINGFIELD-HOLYOKE MA

POSSIBLE STORE-TYPE COLUMNS
- store_code_uc
- retailer_code
- channel_code
- sto